# STHGCN native adapter 验证

目标：把 `sthgcn_mse` 从 graph feature Ridge proxy 推进到 `src/model` 的 native graph adapter。Notebook 只保留实验流程、外部参考 URL、matrix sidecar 和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。

## 外部参考边界

- GitHub: `https://github.com/midas-research/sthgcn-icdm`，核对 STHGCN 公开代码边界、论文题名、ICDM 2020 引用和运行依赖。
- IEEE Computer Society: `https://www.computer.org/csdl/proceedings-article/icdm/2020/831600a482/1r54HpDraaQ`，核对 `Spatiotemporal Hypergraph Convolution Network for Stock Movement Forecasting` 的论文元数据。
- arXiv: `https://arxiv.org/abs/1709.04875`，参考 STGCN 中 temporal convolution 与 graph convolution 组合的公开结构方向。

本实验不复制论文全文或第三方源码；只把“时间卷积 + 超图传播”的结构假设沉入本仓库 compact SOXL feature-node adapter。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.registry import QUANTBENCH_MODEL_NAMES
from src.model.training import fit_estimator_for_model, model_adapter_name

EXPERIMENT_DIR = REPO_ROOT / 'research' / 'sthgcn_native_adapter_v1'
CARD_DIR = EXPERIMENT_DIR / 'matrix_card'
MODEL_NAMES = ('ridge_mse', 'hypergraph_mse', 'sthan_mse', 'thgnn_mse', 'sthgcn_mse', 'market_graph_mse')

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)
output_tables = {
    'predictions': outputs.predictions,
    'daily_backtest': outputs.daily,
    'model_run_ledger': outputs.model_run_ledger,
    'regime_metrics': outputs.regime_metrics,
    'window_metrics': outputs.window_metrics,
    'window_regime_metrics': outputs.window_regime_metrics,
    'model_comparison': outputs.model_comparison,
    'qlib_risk_metrics': outputs.qlib_risk_metrics,
    'model_effect_deltas': outputs.model_effect_deltas,
}
list(output_tables)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


['predictions',
 'daily_backtest',
 'model_run_ledger',
 'regime_metrics',
 'window_metrics',
 'window_regime_metrics',
 'model_comparison',
 'qlib_risk_metrics',
 'model_effect_deltas']

In [2]:
frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame)
train = dataset.frame[dataset.frame['date'] <= pd.Timestamp('2026-04-30')]
feature_columns = dataset.feature_columns[:8]

estimator, constant_prediction = fit_estimator_for_model(
    'sthgcn_mse',
    train=train,
    feature_columns=feature_columns,
)
assert constant_prediction is None

comparison = outputs.model_comparison.set_index('model')
ledger = outputs.model_run_ledger.set_index('model')
capability = model_capability_table(('sthgcn_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary(QUANTBENCH_MODEL_NAMES)

summary = {
    'schema_version': 'research.sthgcn_native_adapter_v1',
    'status': 'completed',
    'models': ['sthgcn_mse'],
    'adapter': model_adapter_name('sthgcn_mse'),
    'estimator': estimator.__class__.__name__,
    'sthgcn': {
        'node_count': int(estimator.node_count_),
        'feature_count': int(estimator.feature_count_),
        'temporal_channel_count': int(estimator.temporal_channel_count_),
        'hyperedge_count': int(estimator.hyperedge_count_),
        'graph_convolution_count': int(estimator.graph_convolution_count_),
        'hidden_size': int(estimator.hidden_size),
        'incidence_shape': list(estimator.incidence_matrix_.shape),
        'hypergraph_propagation_shape': list(estimator.hypergraph_propagation_.shape),
        'temporal_convolution_weight_shape': list(estimator.temporal_convolution_weights_.shape),
        'return_pct': float(comparison.loc['sthgcn_mse', 'return_pct']),
        'sharpe': float(comparison.loc['sthgcn_mse', 'sharpe']),
        'max_drawdown_pct': float(comparison.loc['sthgcn_mse', 'max_drawdown_pct']),
    },
    'paper_fidelity': str(capability['paper_fidelity']),
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': int(
        sum(fidelity['fidelity_counts'].get(key, 0) for key in ('native_graph', 'native_library', 'native_sequence', 'native_torch'))
    ),
    'full_matrix_proxy_model_count': int(fidelity['fidelity_counts']['proxy_feature_adapter']),
    'remaining_proxy_models': fidelity['proxy_models'],
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'matrix_models': sorted(outputs.model_run_ledger['model'].astype(str).tolist()),
    'ledger_trained': bool(ledger.loc['sthgcn_mse', 'trained']),
    'ledger_backtested': bool(ledger.loc['sthgcn_mse', 'backtested']),
    'analysis_layers': sorted(
        key for key, value in output_tables.items() if hasattr(value, 'shape') and key not in {'predictions', 'daily_backtest'}
    ),
    'analysis_table_counts': {
        key: int(value.shape[0]) for key, value in output_tables.items() if hasattr(value, 'shape')
    },
    'external_references': [
        {
            'source_id': 'github:midas-research/sthgcn-icdm',
            'url': 'https://github.com/midas-research/sthgcn-icdm',
            'usage': '核对 STHGCN 公开代码边界、论文题名、ICDM 2020 引用和依赖；本仓库未复制其源码。',
        },
        {
            'source_id': 'doi:10.1109/ICDM50108.2020.00057',
            'url': 'https://www.computer.org/csdl/proceedings-article/icdm/2020/831600a482/1r54HpDraaQ',
            'usage': '核对 STHGCN 论文元数据；本实现只保留 compact temporal-conv plus hypergraph propagation adapter。',
        },
        {
            'source_id': 'arxiv:1709.04875',
            'url': 'https://arxiv.org/abs/1709.04875',
            'usage': '参考 STGCN 中 temporal convolution 和 graph convolution 组合的公开结构方向。',
        },
    ],
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
        'qlib_risk_metrics': str((CARD_DIR / 'qlib_risk_metrics.csv').relative_to(REPO_ROOT)),
        'model_effect_deltas': str((CARD_DIR / 'model_effect_deltas.csv').relative_to(REPO_ROOT)),
        'model_regime_window_diagnostics': str((CARD_DIR / 'model_regime_window_diagnostics.csv').relative_to(REPO_ROOT)),
    },
}

summary_path = EXPERIMENT_DIR / 'notebook_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
summary

{'schema_version': 'research.sthgcn_native_adapter_v1',
 'status': 'completed',
 'models': ['sthgcn_mse'],
 'adapter': 'sthgcn_spatio_temporal_hypergraph_mse_v1',
 'estimator': 'STHGCNGraphRegressor',
 'sthgcn': {'node_count': 8,
  'feature_count': 8,
  'temporal_channel_count': 4,
  'hyperedge_count': 11,
  'graph_convolution_count': 2,
  'hidden_size': 8,
  'incidence_shape': [8, 11],
  'hypergraph_propagation_shape': [8, 8],
  'temporal_convolution_weight_shape': [8, 1, 2],
  'return_pct': 24.90435593735487,
  'sharpe': 2.344077588554966,
  'max_drawdown_pct': 32.28358198342357},
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 29,
 'full_matrix_proxy_model_count': 3,
 'remaining_proxy_models': ['extra_trees_rank',
  'ridge_ic',
  'sequence_lag_ridge_mse'],
 'matrix_card': 'research/sthgcn_native_adapter_v1/matrix_card',
 'matrix_models': ['hypergraph_mse',
  'market_graph_mse',
  'ridge_mse',
  'sthan_mse',
  'sthgcn_mse',
  'thgnn_mse'],
 '

In [3]:
assert summary['adapter'] == 'sthgcn_spatio_temporal_hypergraph_mse_v1'
assert summary['estimator'] == 'STHGCNGraphRegressor'
assert summary['paper_fidelity'] == 'native_graph'
assert summary['is_proxy'] is False
assert summary['full_matrix_native_model_count'] == 29
assert summary['full_matrix_proxy_model_count'] == 3
assert summary['remaining_proxy_models'] == ['extra_trees_rank', 'ridge_ic', 'sequence_lag_ridge_mse']
assert summary['ledger_trained'] is True
assert summary['ledger_backtested'] is True
assert summary['sthgcn']['feature_count'] == 8
assert summary['sthgcn']['node_count'] == 8
assert summary['sthgcn']['temporal_channel_count'] == 4
assert summary['sthgcn']['hyperedge_count'] >= 4
assert summary['sthgcn']['graph_convolution_count'] == 2
assert summary['sthgcn']['incidence_shape'] == [8, summary['sthgcn']['hyperedge_count']]
assert summary['sthgcn']['hypergraph_propagation_shape'] == [8, 8]
assert summary['sthgcn']['temporal_convolution_weight_shape'] == [8, 1, 2]
print('sthgcn_mse native adapter 断言通过。')

sthgcn_mse native adapter 断言通过。


## 中文结论

- 做了什么：在 `src/model.graph` 增加 `STHGCNGraphRegressor`，把 `sthgcn_mse` 从 graph feature proxy 升级为 `native_graph`。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`hypergraph_mse`、`sthan_mse`、`thgnn_mse`、`sthgcn_mse`、`market_graph_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- regime 差异：局部 matrix 的 regime/window 差异进入 sidecar；跨全模型对比继续以 `research/quantbench_market_regime_analysis_v1` 为统一证据面。
- 是否晋升：只晋升模型实现能力，不直接晋升生产策略；生产晋升仍需要更长 walk-forward、压力窗口和 paper/sim 验证。
- 风险：`STHGCNGraphRegressor` 是 SOXL feature-node compact adapter，不声称完整复现 ICDM 2020 STHGCN；当前超图来自特征分组和相邻特征关系，后续需要真实 support-universe 关系图。
- 下一步：全模型注册表当前 native=29、proxy=3；剩余 proxy 为 `extra_trees_rank`、`ridge_ic`、`sequence_lag_ridge_mse`，后续按单模型 notebook 继续推进。